# DMR distribution on ideogram

## Loading necessary libraries

In [1]:
# conda env : r_plot
require(RIdeogram)
library(ggplot2)
library(ggExtra)
library(dplyr)
library(rsvg)
library(ggrepel)
library(showtext)
font_add_google("Noto Sans")
showtext_auto()
figureDir = "/home/jhu/jkim789/projects_aphill27/projects/pooch/figures"

Loading required package: RIdeogram


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Linking to librsvg 2.58.4

Loading required package: sysfonts

Loading required package: showtextdb



## Load data

In [2]:
fai = read.table("/weka/scratch/jhu/aphill27/references/chm13/chm13v2.0.fa.fai", header = FALSE, col.names = c("Chr", "Length", "Offset", "Linebases", "Linewidth"))
fai = fai %>% select(Chr, Length)

cytoband = read.table("/weka/scratch/jhu/aphill27/references/chm13/chm13v2.0_cytobands_allchrs.bed", header = FALSE, col.names = c("Chr", "Start", "End", "Band", "Stain"))
# get first q in Band column by chromosome 
first_q <- cytoband %>%
  group_by(Chr) %>%
  filter(grepl("q", Band)) %>%
  slice(1) %>%
  ungroup() %>%
  select(Chr, End)

# get last p in Band column by chromosome
last_p <- cytoband %>%
  group_by(Chr) %>%
  filter(grepl("p", Band)) %>%
  slice(n()) %>%
  ungroup() %>%
  select(Chr, Start)

# merge first_q, last_p, and fai to create a human karyotype data frame
human_karyotype <- first_q %>%
  inner_join(last_p, by = "Chr") %>%
  inner_join(fai, by = "Chr") %>%
  select(Chr, Length, Start, End)

human_karyotype['Start_0'] = 0
human_karyotype = human_karyotype %>% select(Chr, Start_0, Length, Start, End)
colnames(human_karyotype) <- c('Chr','Start','End','CE_start','CE_end')

# remove chrY from human_karyotype
human_karyotype = human_karyotype %>% filter(Chr != 'chrY')

# sort by chromosome number
human_karyotype = human_karyotype %>% arrange(as.numeric(gsub("chr", "", Chr)))

human_karyotype %>% head(3)

Warning message:
“There was 1 warning in `arrange()`.
ℹ In argument: `..1 = as.numeric(gsub("chr", "", Chr))`.
Caused by warning:
! NAs introduced by coercion”


Chr,Start,End,CE_start,CE_end
<chr>,<dbl>,<int>,<int>,<int>
chr1,0,248387328,121796048,126300487
chr2,0,242696752,92333543,94673023
chr3,0,201105948,91738002,96415026


In [ ]:
# add several bp for visualization.
padding=500000

dmr <- read.table(
    "/home/jhu/jkim789/projects_aphill27/projects/pooch/pooch_data/05.dss_using_123_pairing_chrX_balance/dmrs.N05_filt.all_chr.1e3.mrg.bed",
    sep = "\t",
    header = TRUE,
    comment.char = "",
    quote = ""
)
# colnames(dmr) <- c("Chr", "Start", "End", "length", "nCG", "meanMethy1", "meanMethy2", "DiffMethyl", "areaStat")
dmr <- dmr[c("chr", "start", "end", "diff_methyl_mean")]
colnames(dmr)<- c("Chr", "Start", "End", "DiffMethyl")
# dmr <- dmr[abs(dmr$DiffMethyl) > 0.05, ]
dmr$name <- paste0(dmr$Chr, ":", dmr$Start %/% 1e6)
dmr$type = ifelse(dmr$DiffMethyl > 0, "Maternal", "Paternal")
dmr %>% group_by(type) %>% summarise(n = n())

dmr <- dmr %>%
  mutate(
    Value = case_when(
      DiffMethyl >= 0.15  ~ 5,
      DiffMethyl > 0  ~ 4,
      DiffMethyl < 0  ~ 3,
      DiffMethyl <= -0.15  ~ 2,
      TRUE            ~ 1
    ),
    category = case_when(
      DiffMethyl >= 0.115  ~ "Strong maternal",
      DiffMethyl > 0  ~ "Moderate maternal",
      DiffMethyl == 0 ~ "zero",
      DiffMethyl < 0  ~ "Moderate paternal",
      DiffMethyl <= -0.15  ~ "Strong paternal",
      TRUE            ~ "Subtle"
    ),
    color = case_when(
      category == "Strong maternal"   ~ "#FF0000",
      category == "Moderate maternal" ~ "#e78080",
      category == "zero"            ~ "#fdfdfd",
      category == "Moderate paternal" ~ "#8484fc",
      category == "Strong paternal"   ~ "#0000FF"
    )
  )

dmr['Start'] = dmr['Start'] - padding
dmr['End'] = dmr['End'] + padding

head(dmr)

type,n
<chr>,<int>
Maternal,432
Paternal,448


,Chr,Start,End,DiffMethyl,name,type,Value,category,color
,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>
1,chr1,2040270,2240331,0.10216964,chr1:2,Maternal,4,Moderate maternal,#e78080
2,chr1,5136010,5336161,0.06087192,chr1:5,Maternal,4,Moderate maternal,#e78080
3,chr1,7257760,7458617,0.14710539,chr1:7,Maternal,4,Strong maternal,#FF0000
4,chr1,9387018,9587072,0.08047695,chr1:9,Maternal,4,Moderate maternal,#e78080
5,chr1,32228963,32429318,0.22765704,chr1:32,Maternal,5,Strong maternal,#FF0000
6,chr1,39284128,39484462,0.13760106,chr1:39,Maternal,4,Strong maternal,#FF0000


In [9]:
idmr <- read.table("/home/jhu/jkim789/projects_aphill27/projects/pooch//findings/Imprinted_DMR_List_V1.LiftedOverToT2T-CHM13v2Hs1.bed",
                  sep='\t', header = FALSE)
colnames(idmr) <- c("Chr", "Start", "End", "Type", "Genes", "Ref")
# print(idmr)
# if Type == "Maternal", color = "red", else if Type == "Paternal", color = "blue"
idmr$color <- ifelse(idmr$Type == "Maternal", "e78080", "8080b2") # maternal = red, paternal = blue
idmr$Shape <- ifelse(idmr$Type == "Maternal", "circle", "box")
idmr <- idmr[c("Type", "Shape", "Chr", "Start", "End", "color")]

head(idmr)

,Type,Shape,Chr,Start,End,color
,<chr>,<chr>,<chr>,<int>,<int>,<chr>
1,Maternal,circle,chr1,11044370,11044544,e78080
2,Maternal,circle,chr1,21120031,21120143,e78080
3,Maternal,circle,chr1,32329102,32329320,e78080
4,Maternal,circle,chr1,39384177,39384444,e78080
5,Maternal,circle,chr1,39427318,39428447,e78080
6,Maternal,circle,chr1,67924206,67929467,e78080


## Plotting

In [10]:
pat_my_colors <- c(
  "#0000FF",  # Strong paternal
  "#8484fc"  # Moderate paterna
)

In [11]:
mat_my_colors <- c(
"#e78080",  # Moderate maternal
  "#FF0000"   # Strong maternal
)
  

In [12]:
output = paste0(figureDir,"/hprc_dmr_ideogram_paternal.svg")

ideogram(karyotype = human_karyotype, 
        overlaid = dmr %>% filter(grepl("paternal", category)), colorset1 = pat_my_colors,
        label = idmr, label_type = "marker",
        output = output)
convertSVG(output, device = "png")

Warning message in checkValidSVG(doc, warn = warn):
“This picture may not have been generated by Cairo graphics; errors may result”


In [13]:
output = paste0(figureDir,"/hprc_dmr_ideogram_maternal.svg")

ideogram(karyotype = human_karyotype, 
        overlaid = dmr %>% filter(grepl("maternal", category)), colorset1 = mat_my_colors,
        label = idmr, label_type = "marker",
        output = output)
convertSVG(output, device = "png")

Warning message in checkValidSVG(doc, warn = warn):
“This picture may not have been generated by Cairo graphics; errors may result”


In [14]:
sessionInfo()

R version 4.5.3 (2026-03-11)
Platform: x86_64-conda-linux-gnu
Running under: Rocky Linux 9.6 (Blue Onyx)

Matrix products: default
BLAS/LAPACK: /weka/home/jhu/jkim789/miniforge3/envs/r_plot/lib/libopenblasp-r0.3.34.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: America/New_York
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] showtext_0.9-8  showtextdb_3.0  sysfonts_0.8.9  ggrepel_0.9.8  
[5] rsvg_2.6.1      dplyr_1.2.1     ggExtra_0.11.0  ggplot2_4.0.3  
[9] RIdeogram_0.2.2

loaded via a namespace (and not attached):
 [1] generics_0.1.4     tidyr_1.3.2        jpeg_0.1-11        digest_